In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Tutorial 15. Three-sphere swimmer with a passive elastic arm

## Background

Montino & DeSimone (2015) studied a variant of the Najafi–Golestanian 
three-sphere swimmer in which one of the two rods is replaced by a Hookean spring.  
Spheres 0 (middle), 1 (left), 2 (right) sit on a line at lab positions 
$x_1 < x_0 < x_2$.  The right arm has its length actively prescribed,

$$L_2(t)=x_2 - x_0 = l_2\bigl[1+\varepsilon\sin(\omega t)\bigr],$$

while the left arm $L_1=x_0 -x_1$ is a *passive* degree of freedom
evolving under the spring restoring force $-k(L_1-l_1)$, with $l_1$ the rest
length of the spring. In addition, the hydrodynamic drag apply on the three beads.  

After non-dimensionalisation the dynamics is governed by these four parameters 
(one dynamic, three geometric)

$$\Omega=\frac{\omega\mu (l_1+l_2)}{k}, \quad \frac{l_1}{l_2}, \quad \frac{a}{l_2}, \quad \varepsilon,$$

where we take $l_2$ as the characteristic length and $a$ is the sphere radius.
The leading-order asymptotic in $\varepsilon$ predicts that the net
displacement per period $|x_0|$ is maximized at $\Omega = G_0$, where $G_0$
is the function of the geometry given in Eq. (37) of Montino & DeSimone (2015).
This tutorial fixes $\mu=\omega=l_2=1$, with $l_1/l_2=1$ and $a/l_2=0.05$ by default,
so $\Omega = (l_1+l_2)/k$ and the optimum becomes $k^\star = (l_1+l_2)/G_0$.

Our goal in this tutorial is to:
1. build the body, 
2. plot the shape-space loops $L_1$ vs $L_2$ at three frequencies,
3. scan $|x|/\varepsilon^2$ across $\Omega$ and overlay the leading-order asymptotics, 
4. use `FlowBodyOptimizer` to recover $k^\star$ from a single scalar optimisation, 
5. re-optimise jointly over $(k,l_1/l_2,a_0)$, 
6. overlay the leading-order loop on the numerical one at the optimum for several $\varepsilon$ (Fig. 4).

### Imports

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax

import softmobility as sm
from softmobility.classes import figstyle

jax.config.update("jax_enable_x64", True)
figstyle.apply()
np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

FIGDIR = "figures"

## 1. Defining the swimmer body

Three spheres on the x-axis.  Sphere 0 (left, extremal) is the body
anchor; the spring acts between spheres 0 and 1; sphere 2 (right) is
prescribed kinematically via the `time` symbol inside the YAML.  Only
$L_1$ is a passive DOF — $L_2(t)$ is built directly into sphere 2's
position expression, so we do not need `clamp_dofs_fn`.

In [ ]:
yaml_data = """
dof_names: [L]
design_names: [k, l1, a, eps]

# Convention (matches existing softmobility three-sphere swimmer YAML idiom):
#   sphere 0 = MIDDLE  bead  -> body anchor at body-frame [0, 0, 0]
#   sphere 1 = LEFT    bead  -> body-frame [-(l1 + L0), 0, 0]  (passive, spring)
#   sphere 2 = RIGHT   bead  -> body-frame [l2 * (1 + eps*sin(time), 0, 0]
# L0 is the *deviation* of L1 from its rest length l1.
defaults:
  k:   1.0           # spring stiffness
  L0:  0.0           # initial spring deviation = 0 (at rest)
  l1:  1.0           # spring rest length
  eps: 1.0           # active-arm amplitude / l2
  a0:  0.05          # MIDDLE sphere radius
  a1:  0.05          # LEFT (spring) sphere radius
  a2:  0.05          # RIGHT (active)  sphere radius
# constants (part of defaults, noted with underscores to indicate they are not design parameters):
  _l2_:   1.0        # reference length scale (= l2)
  _omega_: 1.0       # reference angular frequency

spheres:
  - radius: a0
    force: [-k * L0 * _l2_, 0, 0]               # spring on middle, pulls -x when stretched
  - radius: a1
    position: [-(l1 + L0) * _l2_, 0, 0]
    force: [k * L0 * _l2_, 0, 0]                # spring on left, pulls +x when stretched
  - radius: a2
    position: [_l2_ * (1 + eps * sin(_omega_ * time)), 0, 0]
"""

In [ ]:
mybody = sm.SoftBody(yaml_data, verbose=False)
print(repr(mybody))
print("design_variables:", mybody.design_variables)
print("design_defaults: ", np.asarray(mybody.design_defaults))

## 2. Body geometry

Three red translucent spheres in body-frame with the spring sketched as a
zig-zag on the $L_1$ segment.  Saved as `figures/fig_swimmer_geometry.pdf`.

In [ ]:
# 2D side-view schematic (the swimmer is 1D, so 2D is enough and reads
# more clearly than a constrained 3D box).
import matplotlib.pyplot as plt

l1_def, l2_def, a_def = 1.0, 1.0, 0.05
x_centres = np.array([-l1_def, 0.0, l2_def])   # left, middle, right
labels = ["sphere 1\n(left)", "sphere 0\n(middle)", "sphere 2\n(right)"]

fig_geom, ax_geom = figstyle.figure(size="full", aspect=4.0)

# Spring zig-zag between sphere 1 (left) and sphere 0 (middle).
n_z = 16
x_zig = np.linspace(x_centres[0] + a_def, x_centres[1] - a_def, n_z)
y_zig = np.zeros(n_z)
y_zig[1:-1] = 0.10 * np.array([1, -1] * ((n_z - 2) // 2 + 1))[:n_z - 2]
ax_geom.plot(x_zig, y_zig, color=figstyle.COLORS["black"], linewidth=1.2)

# Solid rod between sphere 0 (middle) and sphere 2 (right).
ax_geom.plot(
    [x_centres[1] + a_def, x_centres[2] - a_def], [0, 0],
    color=figstyle.COLORS["black"], linewidth=1.2,
)

# Beads
for xc, lab in zip(x_centres, labels, strict=True):
    circ = plt.Circle((xc, 0), a_def,
                      facecolor=figstyle.COLORS["red_25"],
                      edgecolor=figstyle.COLORS["red"], linewidth=1.0)
    ax_geom.add_patch(circ)
    ax_geom.text(xc, -0.30, lab, ha="center", va="top", fontsize=9)

# Annotate arm lengths
ax_geom.annotate("", xy=(0.0, 0.30), xytext=(-l1_def, 0.30),
                 arrowprops=dict(arrowstyle="<->", color=figstyle.COLORS["black"], lw=0.8))
ax_geom.text(-l1_def/2, 0.40, r"$L_1$",
             ha="center", va="bottom", fontsize=9)
ax_geom.annotate("", xy=(l2_def, 0.30), xytext=(0.0, 0.30),
                 arrowprops=dict(arrowstyle="<->", color=figstyle.COLORS["black"], lw=0.8))
ax_geom.text(l2_def/2, 0.40, r"$L_2(t)$",
             ha="center", va="bottom", fontsize=9)

ax_geom.set_xlim(-1.4, 1.4)
ax_geom.set_ylim(-0.5, 0.5)
ax_geom.set_aspect("equal")
ax_geom.set_xticks([])
ax_geom.set_yticks([])
for s in ax_geom.spines.values():
    s.set_visible(False)

figstyle.save(fig_geom, "fig_swimmer_geometry", figdir=FIGDIR)

## 3. Shape-space loops at three frequencies

We scan $\Omega = (l_1+l_2)/k \in \{0.1, 1, 10\}$ by setting $k \in \{20,2,0.2\}$,
hold $\varepsilon = 0.5$, run 6 periods, discard the first 4 to settle to the
limit cycle, and plot $L_1$ vs $L_2$ in the same axes (paper Fig. 2).

In [ ]:
rollout = sm.FlowBodyRollout(soft_body=mybody, flow=sm.no_flow())

# Build a helper that runs the rollout with a given design and returns
# the trajectory of (L1, L2) over the last `n_keep_periods`.
DESIGN_NAMES = list(mybody.design_variables)
DEFAULTS = jnp.asarray(mybody.design_defaults)

def design_with(**kwargs):
    d = DEFAULTS
    for name, value in kwargs.items():
        idx = DESIGN_NAMES.index(name)
        d = d.at[idx].set(value)
    return d


PERIOD = 2.0 * np.pi
N_PER_PERIOD = 200
DT = PERIOD / N_PER_PERIOD


def run_loop(design, n_settle=4, n_keep=2):
    n_total = (n_settle + n_keep) * N_PER_PERIOD
    positions, _, dofs = rollout.rollout(dt=DT, n_steps=n_total, design=design)
    positions = np.asarray(positions)
    dofs = np.asarray(dofs)
    # Time grid; rollout returns post-step samples
    t = (np.arange(n_total) + 1) * DT
    keep = slice(n_settle * N_PER_PERIOD, None)
    t_k = t[keep]
    # L_0 dof is the deviation of L_1 from rest l1; convert to absolute lengths.
    l1 = float(design[DESIGN_NAMES.index("l1")])
    l2 = 1.0  # _l2_ is the characteristic length
    eps = float(design[DESIGN_NAMES.index("eps")])
    L1 = l1 + dofs[keep, 0]
    L2 = l2 * (1.0 + eps * np.sin(t_k))
    return t_k, L1, L2, positions[keep], dofs[keep]


loops = {}
for k_val, label, colour in [
    (20.0,  r"$\Omega = 0.1$",  figstyle.COLORS["black"]),
    (2.0,   r"$\Omega = 1$",    figstyle.COLORS["red"]),
    (0.2,   r"$\Omega = 10$",   figstyle.COLORS["blue"]),
]:
    d = design_with(k=k_val, eps=0.5)
    _, L1, L2, _, _ = run_loop(d)
    loops[k_val] = (L1, L2, label, colour)
    print(f"k={k_val:>5.2g}, Omega={2.0/k_val:>5.2g}: "
          f"L1 in [{L1.min():.3f}, {L1.max():.3f}]")

fig_loops, ax_loops = figstyle.figure(size="half", aspect=1.4)
for L1, L2, label, colour in loops.values():
    ax_loops.plot(L2, L1, color=colour, linewidth=1.5, label=label)
ax_loops.set_xlabel(r"$L_2$")
ax_loops.set_xlim(0.2, 1.8)
ax_loops.set_xticks([0.5, 1.0, 1.5])
ax_loops.set_ylabel(r"$L_1$")
ax_loops.set_yticks([0.8, 0.9, 1.0, 1.1, 1.2])
# ax_loops.legend(frameon=False, loc="lower left")

figstyle.add_label(ax_loops, position=(1.75, 1.02), text=r"$\Omega = 0.1$",
             color=figstyle.COLORS["black"], anchor="bottom left",
             offset=(0, 0.05), name="Omega_0.1")
figstyle.add_label(ax_loops, position=(1.4, 1.1), text=r"$\Omega = 1$",
             color=figstyle.COLORS["red"], anchor="bottom left",
             offset=(0, 0.05), name="Omega_1")
figstyle.add_label(ax_loops, position=(1.05, 1.18), text=r"$\Omega = 10$",
             color=figstyle.COLORS["blue"], anchor="bottom left",
             offset=(0, 0.05), name="Omega_10")
figstyle.save(fig_loops, "fig_swimmer_shape_loops", figdir=FIGDIR)

## 4. Leading-order theory and displacement scan

Define the analytical functions $F_0, G_0$ (paper Eqs. 36–37). Scan $k$ over a log-spaced grid, compute the per-period displacement $x_0$ at small $\varepsilon = 0.3$, and overlay the leading-order $|x_0|/\varepsilon^2$ vs $\Omega$ (Eq. 44).

In [ ]:
def F0_fn(l1, l2, a):
    inv = 1.0/(2*l2) - 1.0/(3*a)
    return (1.0/inv) * (1.0/(4*l2) + 1.0/(4*l1) - 1.0/(4*(l1+l2)) - 1.0/(6*a))


def G0_fn(l1, l2, a):
    l = l1 + l2
    inv = 1.0/(2*l2) - 1.0/(3*a)
    bracket = (1.0/(4*l2) + 1.0/(4*l1) - 1.0/(4*(l1+l2)) - 1.0/(6*a))**2
    return (l/np.pi) * ((1.0/(3*a) - 1.0/(2*l1)) + (1.0/inv) * bracket)


def A_fn(Omega, F0, G0):
    return Omega**2 / (Omega**2 + G0**2) * F0


def B_fn(Omega, F0, G0):
    return Omega / (Omega**2 + G0**2) * G0 * F0


L1_DEF, L2_DEF, A_DEF = 1.0, 1.0, 0.05
F0 = F0_fn(L1_DEF, L2_DEF, A_DEF)
G0 = G0_fn(L1_DEF, L2_DEF, A_DEF)
L_TOT = L1_DEF + L2_DEF
LAM1 = L1_DEF / L_TOT
LAM2 = L2_DEF / L_TOT
C_COEF = (A_DEF * L_TOT / 6.0) * (1.0/L1_DEF**2 + 1.0/L2_DEF**2 - 1.0/L_TOT**2)
print(f"F0 = {F0:.4f}")
print(f"G0 = {G0:.4f}    (theoretical k* = (l1+l2)/G0 = {L_TOT/G0:.4f})")
print(f"C  = {C_COEF:.4f}")

In [ ]:
N_SCAN = 30
omega_scan = np.logspace(-1, 1.4, N_SCAN)     # Omega in [0.1, ~25]
k_scan = L_TOT / omega_scan                   # Omega = (l1+l2)/k
eps_scan = 0.1
n_settle_scan = 4
n_one = N_PER_PERIOD


@jax.jit
def run_settle_and_period(design):
    pos_s, _, dofs_s = rollout.rollout(dt=DT, n_steps=n_settle_scan*n_one,
                                       design=design)
    pos1, _, _ = rollout.rollout(
        dt=DT, n_steps=n_one, design=design,
        init_position=pos_s[-1], init_orientation=jnp.zeros(3),
        init_dofs=dofs_s[-1],
    )
    return pos1[-1, 0] - pos_s[-1, 0]


X_num = np.zeros(N_SCAN)
for i, k_val in enumerate(k_scan):
    d = design_with(k=float(k_val), eps=eps_scan)
    X_num[i] = float(run_settle_and_period(d))

# Leading-order: X / l = -2*pi*lam2**2 * C * B(Omega) * eps**2  (in units of l = l1+l2)
# X_num is in units of l2, so divide by L_TOT to compare with the M&D dimensionless form.
X_LO = -2.0 * np.pi * LAM2**2 * C_COEF * B_fn(omega_scan, F0, G0)  # without eps^2 since we divide

fig_scan, ax_scan = figstyle.figure(size="half", aspect=1.4)
ax_scan.plot(omega_scan, np.abs(X_num)/L_TOT/eps_scan**2,
             color=figstyle.COLORS["blue"], linewidth=1.5, label="simulation")
ax_scan.plot(omega_scan, np.abs(X_LO),
             color=figstyle.COLORS["blue"], marker="o", markersize=3, linestyle="none", label="asympt.")
ax_scan.axvline(G0, color=figstyle.COLORS["grey"], linewidth=1.5,
                linestyle=":", label=r"$G_0$")
ax_scan.set_xscale("log")
ax_scan.set_xlabel(r"$\Omega$")
ax_scan.set_ylabel(r"$|x_0|/\varepsilon^2$")
ax_scan.set_yticks([0, 0.005, 0.010])
ax_scan.legend(frameon=False, loc="upper left")

figstyle.save(fig_scan, "fig_swimmer_displacement", figdir=FIGDIR)
print(f"argmax numerical |X| at Omega = {omega_scan[np.argmax(np.abs(X_num))]:.3f}, "
      f"theoretical G0 = {G0:.3f}")

## 5. Optimization

We use `FlowBodyOptimizer` to recover the optimal spring stiffness numerically.  

In [ ]:
K_IDX = DESIGN_NAMES.index("k")
L1_IDX = DESIGN_NAMES.index("l1")
A1_IDX = DESIGN_NAMES.index("a1")        # radius of LEFT (extremal) sphere
EPS_IDX = DESIGN_NAMES.index("eps")
SCAN_EPS = 0.1                          # use small eps so leading-order is meaningful

DEFAULTS_SMALL = DEFAULTS.at[EPS_IDX].set(SCAN_EPS)   # baseline for objectives


def displacement_per_period(myrollout, full_design):
    pos_s, _, dofs_s = myrollout.rollout(
        dt=DT, n_steps=4*N_PER_PERIOD, design=full_design)
    pos1, _, _ = myrollout.rollout(
        dt=DT, n_steps=N_PER_PERIOD, design=full_design,
        init_position=pos_s[-1], init_orientation=jnp.zeros(3),
        init_dofs=dofs_s[-1],
    )
    return pos1[-1, 0] - pos_s[-1, 0]


def objective_A(myrollout, design):
    full_design = DEFAULTS_SMALL.at[K_IDX].set(design[0])
    return -jnp.abs(displacement_per_period(myrollout, full_design))   # minimise -> maximise |X|


opt_A = sm.FlowBodyOptimizer(rollout, objective_A, optimizer=optax.adam(5e-2))
result_A = opt_A.run(
    init_design=jnp.array([1.0]),
    n_steps=80, print_every=20,
    clip_min=jnp.array([0.05]),
    clip_max=jnp.array([20.0]),
    maximize=False,
)
k_opt_A = float(result_A[0])
k_theory = L_TOT / G0
print(f"\nPass A (eps={SCAN_EPS}): k_opt = {k_opt_A:.4f},  k_theory = (l1+l2)/G0 = {k_theory:.4f},  "
      f"rel.err = {abs(k_opt_A - k_theory)/k_theory*100:.2f}%")

## 6. Leading-order overlay at the optimum

Fix $k = (l_1+l_2)/G_0$ so $\Omega = G_0$ exactly.  Vary $\varepsilon$ over
$\{0.3, 0.5, 0.7\}$.  For each $\varepsilon$, overlay the
numerical shape-space loop (solid line) with the leading-order steady
state (dashed line).

In [ ]:
k_star = L_TOT / G0
EPSILONS = [0.3, 0.5, 0.7]

fig_lo, ax_lo = figstyle.figure(size="half", aspect=1.4)
errors = {}
for eps in EPSILONS:
    d = design_with(k=k_star, eps=eps)
    _, L1_num, L2_num, _, _ = run_loop(d, n_settle=5, n_keep=1)
    # Sort by time to plot a closed curve; one period is already closed.
    ax_lo.plot(L2_num, L1_num,
               color=figstyle.COLORS["blue"], linewidth=1.0)
    # Leading-order steady state
    tau = np.linspace(0, 2*np.pi, 32) 
    A = A_fn(G0, F0, G0)
    B = B_fn(G0, F0, G0)
    x1_LO = -A * LAM2 * np.sin(tau) - B * LAM2 * np.cos(tau)   # dimensionless x1
    # Convert to dimensional L1: L1 = l1 + l * eps * x1
    L1_LO = L1_DEF + L_TOT * eps * x1_LO
    L2_LO = L2_DEF * (1 + eps * np.sin(tau))
    ax_lo.plot(L2_LO, L1_LO,
               color=figstyle.COLORS["blue"], linestyle="none", marker="o", markersize=3)

# Add legend entries (one per style; ε labels via annotations on outer loop)
ax_lo.plot([], [], color=figstyle.COLORS["blue"], linewidth=1.0,
           label="rollout")
ax_lo.plot([], [], color=figstyle.COLORS["black"], linestyle="--",
           linewidth=0.9, label="leading order")
ax_lo.set_xlabel(r"$L_2$")
ax_lo.set_xlim(0.2, 1.8)
ax_lo.set_xticks([0.5, 1.0, 1.5])
ax_lo.set_ylabel(r"$L_1$")
ax_lo.set_yticks([0.8, 0.9, 1.0, 1.1, 1.2])

figstyle.save(fig_lo, "fig_swimmer_leading_order", figdir=FIGDIR)

## 7. Full optimization

We now fix the active link $L2$ and the radius of the sphere 0 and 2. But the other 
parameters are free and optimized (the leftmost sphere, the spring stiffness and rest length).
In each pass we optimize only a*subset* of the design parameters by composing the full 
design array inside the objective from a small variable vector plus the fixeddefaults.

In [ ]:
# Pass B is run at a larger amplitude than Pass A so the optimal shape loop is
# visible in the L_2-L_1 plane below (Pass A stays at eps=0.1 to keep its
# comparison with the leading-order k* = (l1+l2)/G_0 sharp).
EPS_OPT_B = 0.5
DEFAULTS_OPT_B = DEFAULTS.at[EPS_IDX].set(EPS_OPT_B)


def objective_B(myrollout, design):
    full = DEFAULTS_OPT_B \
        .at[K_IDX].set(design[0]) \
        .at[L1_IDX].set(design[1]) \
        .at[A1_IDX].set(design[2])
    return -jnp.abs(displacement_per_period(myrollout, full))


opt_B = sm.FlowBodyOptimizer(rollout, objective_B, optimizer=optax.adam(2e-2))
result_B = opt_B.run(
    init_design=jnp.array([k_opt_A, 1.0, 0.05]),  # k, l1, a1
    n_steps=300, print_every=50,
    clip_min=jnp.array([0.05, 0.15, 0.01]),  # let l1 explore down to β = 0.08
    clip_max=jnp.array([20.0, 2.0,  0.5]),
    maximize=False,
)
k_optB, l1_optB, a1_optB = (float(x) for x in result_B)
G0_B = G0_fn(l1_optB, L2_DEF, a1_optB)
# Evaluate |X| at the recovered optimum (eps = EPS_OPT_B for both passes)
full_B = DEFAULTS_OPT_B.at[K_IDX].set(k_optB).at[L1_IDX].set(l1_optB).at[A1_IDX].set(a1_optB)
X_B = float(displacement_per_period(rollout, full_B))
print(f"\nPass B (eps={EPS_OPT_B}): k = {k_optB:.4f}, l1 = {l1_optB:.4f}, a1 (left) = {a1_optB:.4f}")
print(f"  resulting Omega = {(l1_optB + L2_DEF)/k_optB:.4f},  G0(l1*,l2,a1*) = {G0_B:.4f}")
print(f"  beta = l1/l2 = {l1_optB/L2_DEF:.3f}  "
      f"(paper Fig. 8 leading-order predicts beta ~ 0.07 for uniform a; "
      f"we optimise a1 too so the optimum shifts)")
X_A = float(displacement_per_period(rollout, DEFAULTS_OPT_B.at[K_IDX].set(k_opt_A)))
print(f"  |X(opt B)| / |X(opt A)|  = {abs(X_B)/abs(X_A):.2f}   (both evaluated at eps={EPS_OPT_B})")

## 8. Geometry and shape-space loop of the fully-optimized swimmer

The Pass-B optimum shrinks the passive arm to roughly $l_1^\star/l_2 \approx 0.13$
while leaving the active arm at $l_2 = 1$.  Below we draw the resulting
geometry at the same scale as Section 2, then overlay the shape-space loop of
the fully-optimized swimmer on that of the default-geometry swimmer at its own
optimal frequency (Pass A, $\Omega \approx G_0$).  Both loops are plotted
against $L_1 - L_1^{\rm rest}$ on the ordinate so they can be compared
directly even though their absolute $L_1$ levels differ by almost an order of
magnitude.

In [ ]:
# 2D schematic of the fully-optimized swimmer, drawn at the same x-scale
# as fig_swimmer_geometry so the asymmetry l1* << l2 is immediately visible.
x_centres_opt = np.array([-l1_optB, 0.0, l2_def])     # left, middle, right
radii_opt = [a1_optB, a_def, a_def]                   # only a1 (left) is optimized
labels_opt = ["sphere 1\n(left)", "sphere 0\n(middle)", "sphere 2\n(right)"]

fig_geom_opt, ax_geom_opt = figstyle.figure(size="full", aspect=4.0)

# Spring zig-zag on the (very short) left arm — fewer teeth and smaller
# amplitude than the default-geometry plot, since we only have ~l1* - 2a of
# horizontal room.
n_z = 6
x_zig = np.linspace(x_centres_opt[0] + a1_optB,
                    x_centres_opt[1] - a_def, n_z)
y_zig = np.zeros(n_z)
y_zig[1:-1] = 0.05 * np.array([1, -1] * ((n_z - 2) // 2 + 1))[:n_z - 2]
ax_geom_opt.plot(x_zig, y_zig, color=figstyle.COLORS["black"], linewidth=1.2)

# Solid rod between sphere 0 (middle) and sphere 2 (right).
ax_geom_opt.plot(
    [x_centres_opt[1] + a_def, x_centres_opt[2] - a_def], [0, 0],
    color=figstyle.COLORS["black"], linewidth=1.2,
)

# Beads
for xc, rad, lab in zip(x_centres_opt, radii_opt, labels_opt, strict=True):
    circ = plt.Circle((xc, 0), rad,
                      facecolor=figstyle.COLORS["red_25"],
                      edgecolor=figstyle.COLORS["red"], linewidth=1.0)
    ax_geom_opt.add_patch(circ)

ax_geom_opt.set_xlim(-1.4, 1.4)
ax_geom_opt.set_ylim(-0.5, 0.5)
ax_geom_opt.set_aspect("equal")
ax_geom_opt.set_xticks([])
ax_geom_opt.set_yticks([])
for s in ax_geom_opt.spines.values():
    s.set_visible(False)

figstyle.save(fig_geom_opt, "fig_swimmer_geometry_optimized", figdir=FIGDIR)

## References

R. Golestanian and A. Ajdari, *Analytic results for the three-sphere
swimmer at low Reynolds number*, Phys. Rev. E **77**, 036308 (2008).

A. Montino and A. DeSimone, *Three-sphere low-Reynolds-number swimmer with
a passive elastic arm*, Eur. Phys. J. E **38**, 42 (2015).

A. Najafi and R. Golestanian, *Simple swimmer at low Reynolds number*,
Phys. Rev. E **69**, 062901 (2004).